# Organized Play Study
Here we will compare different ways of pairing opponents during large tournaments and see how it might align with our intuition. As, I believe, Twice pointed out, the ultimate goal of OP is a fun experience that people want to participate in again in the future. We can't really accomplish this. A secondary goal would be to attempt to find a winner who has the greatest "skill," whatever that means.

The main question we are facing is how to use the "agent scores" paremeter in the pairing algorithm. This is the pair of numbers given by the total number of agents scored by each player over the course of the match (including the advantage/bonus agent). There are some obvious candidates:
- total agents
- agent differential
- total agent ratio
- agent percentage
- weighted versions of the above

There may be other functions on the "agent sequence." This is just the sequence of agent scores at the end of a match taken in succession. One reason to consider this is that as the tournament progresses, we assume that participants will filter into various levels of skill. Once they have been sorted and are playing other participants of comparable skill, these agent scores will be more relevant than agent scores from the beginning of the tournament.

### A word on skill

For the purposes of this study we have assigned players a skill value. This is some number that we pull out of thin air. We recognize that this is a major flaw. It isn't just the player, it is the deck, it is the matchups, it is the order the cards come in. Unfortunately, this is too complicated. We say this all here to indicate that we recognize the weakness here but doing anything else would probably be not a very good use of time. We do not think that this holds any real value beyond giving some idea of how things might look. This is just a simulation. And probably a bad one... Womp Womp.

## Setup

### Tournaments

Each tournament will consist of a collection of players and a number of rounds that are simulated. The players will be assigned a skill value using a normal distribution with mean 0 and standard deviation 1. Note that each tournament will contain an even number of players by construction. Dealing with "byes" is another issue altogether and we don't want to be distracted by it.

### Rounds

Each round of a tournament will be simulated by pairing players and having them play a best of 2 match. After the match, the standings of the players will be taken into account as well as their agent scores. These will be used to determine the pairings for the subsequent rounds.

### Matches and Games

A match will consist of two games. Each game will be simulated using the skills of the players involved. We follow a version of the Bradley-Terry model for determining which player scores the next agent. After the third agent the game is ended with that player winning. The agent score of the match is the number of agents they won and the number of agents they lost. If this is tied at the end of the match then a final bonus agent is assigned using the same function used in simulating the game.

Note, we do not consider the mill condition in this model. There are multiple reasons for this. One is that it would overcomplicate the model significantly. Another is that we don't foresee mill decks being very prevalent in the tournament scene. These are assumptions, but we feel mill players are at a disadvantage in general in organized play given the time limit on matches.

### Pairings

When pairing players for the next round of the tournament we will first sort players into groups according to their records. We will then pass there sequence of agent scores to various rating functions to order them internally to these groups. If groups do not contain an even number of players we will move the top player up into the next group until all groups have an even number of players.It is possible to test different pairing methodologies. We have focused on pairing players that are adjacent to each other.

### Rating functions

We will use the sequence of agent scores (agents won vs agents lost) to various different rating functions. These are the main focus of the study. We are attempting to compare how different rating functions will impact the sequence of the tournament.

Our hope is to isolate candidate rating functions that optimize certain characteristics.
They are:
- accuracy
- efficiency
- clarity/computability
- agency

Each of these is impossible to actually quantify and measure, but they are good to keep in view.

#### Accuracy

We will measure accuracy by seeing how closely it follows the pre-assigned skill of players. It would be bad if some rating function ended up putting a player with a high skill value in a lower position than a player with a lower skill value very consistently. This is imperfect, but it is something to keep in mind.

As we wish to sort players into groups, we would like to minimize the skill gap between players that face each other. But we consider this to be under the following heading.

#### Efficiency

By efficiency we mean how "quickly" the tournament sorts players into groups of similar skill levels. We measure this by looking by the skill gap between opponents. We recognize that "skill" only really makes sense in this model and isn't a real quantifiable attribute of players in the way we represent it with our model.

#### Clarity and Computability

The rating function used should be something accessible. It should be clear how it is computed and easy to compute. One of the main downsides of Stength of Schedule is how opaque the metric is. Not only should players understand it and have a good idea of where they will place, but the organizers also have to be able to easily implement the algorithm.

#### Agency

Another downside of Strength of Schedule is that players don't feel like they can have any influence over it. We would hope that along with clarity for the players, the chosen metric will give them the feeling that they are able to actually impact their own ranking.3

### Comparison to Strength of Schedule

The standard in Swiss style tournaments is to use Strength of Schedule. This is a very popular method. One major downside is that it is opaque and players can not easily determine where they sit in the rankings. One hope of this "agent advantage" score is that it will give players a greater feeling of agency. As Strength of Schedule is a robust system, we will be comparing our different pairing algorithms to this method.

## Study

We begin by importing the necessary modules and defining some constants.

In [1]:
import sys
from pathlib import Path
from random import Random

sys.path.insert(0, str(
    Path.cwd().parent
    if Path.cwd().name == 'notebooks'
    else Path.cwd()
))

import matplotlib.pyplot as plt

from tournament.generators import make_players, skilled_match
from tournament.engine import run_tournament
from tournament.pairing import make_record_group_pairing
from tournament.standings import compute_records, make_rank_key
from tournament.metrics import (
    standings_skill_correlation,
    mean_skill_gap,
    weighted_skill_gap_score,
    rematch_count,
)
from tournament.presentation import display_stacked_rounds, display_tournament_summary
from tournament.rating import (
    agent_differential,
    agent_ratio,
    agent_total_ratio,
    total_agents_scored,
    weighted_agent_differential,
    weighted_agent_total_ratio,
    linear_weights,
    exponential_weights,
)

N_PLAYERS = [i*2 for i in range(4,21)]
N_ROUNDS  = range(3,9)
SEEDS = [42, 123, 456, 789, 1011, 1213, 1415, 1617, 1819, 2021]+[i for i in range(2023, 2043)]
STRATEGY  = "adjacent"  # within-group pairing order; try "fold" or "strong_weak"
TOUR_CTX = [(i, j, seed) for i in N_PLAYERS for j in N_ROUNDS for seed in SEEDS]



### Player Creation

We begin by creating a collection of players. They will have different skill levels assigned using a normal distribution with mean = 0. We will leave the standard deviation as a knob to be tuned. 

In [2]:
players = {(i, seed): make_players(i, Random(seed)) for i in N_PLAYERS for seed in SEEDS}
all_players = []
for p_set in players.values():
    all_players.extend(p_set)
print(f"Players: {min(N_PLAYERS)}-{max(N_PLAYERS)}  Rounds: {min(N_ROUNDS)}-{max(N_ROUNDS)}  Seeds: {len(SEEDS)}  Strategy: {STRATEGY}")
print()
print(f"  {'Name':<8} {'Skill':>8}")
print(f"Max skill: {max([p.skill for p in all_players])}")
print(f"Min skill: {min([p.skill for p in all_players])}")
print(f"Mean skill: {sum(p.skill for p in all_players) / len(all_players):.3f}")
print(f"Std skill: {((sum((p.skill - sum(p.skill for p in all_players) / len(all_players))**2 for p in all_players) / len(all_players))**0.5):.3f}")

Players: 8-40  Rounds: 3-8  Seeds: 30  Strategy: adjacent

  Name        Skill
Max skill: 96
Min skill: 1
Mean skill: 49.297
Std skill: 14.831


## Rating Functions

All rating functions take an **agent sequence** — a list of `(agents_for, agents_against)` tuples, one per round played — and return a single number. A higher value means a better tiebreaker position within a win/loss bracket.

### Primary functions (full sequence)

| Name | Formula | What it captures |
|---|---|---|
| `agent_differential` | `sum(for - against)` | Net agent advantage, all rounds equal weight |
| `agent_total_ratio` | `sum(for) / (sum(for) + sum(against))` | Share of total agents scored; bounded [0, 1] |
| `agent_ratio` | `sum(for) / sum(against)` | Multiplicative margin; can be ∞ if zero conceded |
| `total_agents_scored` | `sum(for)` | Raw offence only; ignores agents conceded |

### Weighted functions (full sequence, recency bias)

**Weighted functions** apply a per-round multiplier that grows across the tournament.
`linear_weights(n)` goes evenly from 1.0 to 2.0; `exponential_weights(n, base)` multiplies by `base` each round.
Weights are constructed dynamically from `len(seq)` so they always match however many rounds have been played.

| Name | What it captures |
|---|---|
| `weighted_diff (linear)` | `agent_differential` with linearly increasing round weights |
| `weighted_diff (exp 1.2)` | `agent_differential` with exponentially increasing weights |
| `weighted_ratio (linear)` | `agent_total_ratio` with linear weights |

### Drop-off functions (last k rounds only)

Each primary function is also applied to only the **last k rounds** of the sequence (`seq[-k:]`), for k = 2, 3, 4. This gives more credit to recent form and ignores early-tournament noise. When fewer than k rounds have been played the full sequence is used.

---

**To add or swap a rating function**, edit the `RATING_FNS` list in the next cell and re-run from there.

In [3]:
# ── Edit this list to change which rating functions are compared ──────────────
# Each entry is (label, rating_fn) where rating_fn(agent_seq) -> float.

# Primary functions — use the full agent sequence.
_PRIMARY = [
    ("agent_differential",  agent_differential),
    ("agent_total_ratio",   agent_total_ratio),
    ("agent_ratio",         agent_ratio),
    ("total_agents_scored", total_agents_scored),
]

# Drop-off variants — apply each primary function to only the last k rounds.
_DROP_OFF = [
    (f"{label} (last {k})", lambda seq, fn=fn, k=k: fn(seq[-k:]))
    for k in (2, 3, 4)
    for label, fn in _PRIMARY
]

# Weighted variants — full sequence with a per-round recency multiplier.
_WEIGHTED = [
    (
        "weighted_diff (linear)",
        lambda seq: weighted_agent_differential(seq, linear_weights(len(seq))),
    ),
    (
        "weighted_diff (exp 1.2)",
        lambda seq: weighted_agent_differential(seq, exponential_weights(len(seq), base=1.2)),
    ),
    (
        "weighted_ratio (linear)",
        lambda seq: weighted_agent_total_ratio(seq, linear_weights(len(seq))),
    ),
]

RATING_FNS = _PRIMARY + _DROP_OFF + _WEIGHTED

print(f"{len(RATING_FNS)} rating functions registered:")
counter = 0
fn_list = ""
for label, _ in RATING_FNS:
    fn_list += label
    if counter % 3 == 0 and counter != 0:
        fn_list += "\n"
    else:
        fn_list += ", "
    counter +=1
print(fn_list)
#for label, _ in RATING_FNS:
#    print(f"  • {label}")

19 rating functions registered:
agent_differential, agent_total_ratio, agent_ratio, total_agents_scored
agent_differential (last 2), agent_total_ratio (last 2), agent_ratio (last 2)
total_agents_scored (last 2), agent_differential (last 3), agent_total_ratio (last 3)
agent_ratio (last 3), total_agents_scored (last 3), agent_differential (last 4)
agent_total_ratio (last 4), agent_ratio (last 4), total_agents_scored (last 4)
weighted_diff (linear), weighted_diff (exp 1.2), weighted_ratio (linear)



In [ ]:
# move to tournament/study.py
import json
from dataclasses import dataclass
from collections import Counter
from pathlib import Path
from typing import Callable, Sequence

from tournament.models import Tournament


@dataclass(frozen=True)
class Result:
    """One tournament run under a single rating function."""
    label: str
    rating_fn: Callable
    tour: object
    corr: float
    gap: float
    weighted_gap: float
    rematches: int
    seed: int

    def to_dict(self) -> dict:
        return {
            "label": self.label,
            "tour": self.tour.to_dict(),
            "corr": self.corr,
            "gap": self.gap,
            "weighted_gap": self.weighted_gap,
            "rematches": self.rematches,
            "seed": self.seed,
        }

    @classmethod
    def from_dict(cls, d: dict, rating_fn: Callable) -> "Result":
        return cls(
            label=d["label"],
            rating_fn=rating_fn,
            tour=Tournament.from_dict(d["tour"]),
            corr=d["corr"],
            gap=d["gap"],
            weighted_gap=d["weighted_gap"],
            rematches=d["rematches"],
            seed=d["seed"],
        )


def run_comparison(n_players: int, n_rounds: int, seed: int) -> list[Result]:
    """Run one tournament per rating function for a fixed (players, rounds, seed) context.

    Uses the same seed and player pool for every rating function so the only
    difference is the within-group tiebreaker.
    """
    results = []
    for label, fn in RATING_FNS:
        rng = Random(seed)
        tour = run_tournament(
            players[(n_players, seed)],
            n_rounds=n_rounds,
            pairing=make_record_group_pairing(STRATEGY, rating_fn=fn),
            rng=rng,
            match_model=skilled_match,
        )
        results.append(Result(
            label=label,
            rating_fn=fn,
            tour=tour,
            corr=standings_skill_correlation(tour),
            gap=mean_skill_gap(tour),
            weighted_gap=weighted_skill_gap_score(tour),
            rematches=rematch_count(tour),
            seed=seed,
        ))
    return results


def run_mass_comparisons(contexts: Sequence[tuple[int, int, int]]) -> dict[tuple[int, int, int], list[Result]]:
    """Run a comparison for every (players, rounds, seed) context."""
    return {ctx: run_comparison(*ctx) for ctx in contexts}


CACHE_DIR  = Path("../cache")
CACHE_FILE = CACHE_DIR / "big_results.json"


def _results_to_json(results_by_ctx) -> dict:
    """Serialize big_results to a JSON-compatible dict."""
    return {
        f"{ctx[0]},{ctx[1]},{ctx[2]}": [r.to_dict() for r in results]
        for ctx, results in results_by_ctx.items()
    }


def _results_from_json(data: dict) -> dict:
    """Rebuild big_results from a JSON-loaded dict."""
    fn_by_label = dict(RATING_FNS)
    out = {}
    for key, result_dicts in data.items():
        ctx = tuple(int(x) for x in key.split(","))
        out[ctx] = [Result.from_dict(d, fn_by_label[d["label"]]) for d in result_dicts]
    return out


def load_or_run(contexts, cache_file=CACHE_FILE):
    """Load cached results if available, otherwise run and cache."""
    if cache_file.exists():
        with cache_file.open("r") as f:
            cached = _results_from_json(json.load(f))
        cached_ctxs = set(cached.keys())
        needed = set(contexts)
        if needed.issubset(cached_ctxs):
            print(f"Loaded {len(needed)} contexts from {cache_file}")
            return {ctx: cached[ctx] for ctx in contexts}
        missing = needed - cached_ctxs
        print(f"Cache has {len(cached_ctxs)} contexts; running {len(missing)} missing...")
        new_results = run_mass_comparisons(missing)
        cached.update(new_results)
        with cache_file.open("w") as f:
            json.dump(_results_to_json(cached), f)
        return {ctx: cached[ctx] for ctx in contexts}
    print(f"No cache at {cache_file}; running full simulation...")
    results = run_mass_comparisons(contexts)
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    with cache_file.open("w") as f:
        json.dump(_results_to_json(results), f)
    print(f"Cached {len(results)} contexts to {cache_file}")
    return results


N_PLAYERS = [i*2 for i in range(4,21)]
N_ROUNDS  = range(3,9)
BIG_SEEDS = [42, 123, 456, 789, 1011, 1213, 1415, 1617, 1819, 2021]+[i for i in range(2023, 2043)]
STRATEGY  = "adjacent"  # within-group pairing order; try "fold" or "strong_weak"
TOUR_CTX = [(i, j, seed) for i in N_PLAYERS for j in N_ROUNDS for seed in BIG_SEEDS]

big_results = load_or_run(TOUR_CTX)


No cache at ../cache/big_results.json; running full simulation...


We wil present the results of the experiment as follows. After simulating the various tournaments, we will look at which rating function performed best for each tournament. This will be based on 3 different metrics. The first is how well performance in the tournament correlates with skill. Next we will look at how each minimizes the (avg) skill gap between players that are matched up. Lastly, we will look at the weighted skill gap where skill gaps in initial rounds contribute less than the gap in later rounds.

Note, this all depends on the pre-assigned skill of each player so these are somewhat arbitrary. Some randomness is introduced so that the games are not completely deterministic so we can not expect these scores to be perfect reflections of the efficacy of the different rating functions. Nonetheless, we still expect some correlation between skill and outcome in the tournament. So take from these what you will.


In [ ]:
# move to tournament/study.py

def top_results(results: Sequence[Result], metric: str, n: int = 3) -> list[Result]:
    """Return the top-n results sorted by metric ('corr', 'gap', or 'weighted_gap')."""
    if metric == "corr":
        return sorted(results, key=lambda r: r.corr, reverse=True)[:n]
    if metric == "gap":
        return sorted(results, key=lambda r: r.gap)[:n]
    if metric == "weighted_gap":
        return sorted(results, key=lambda r: r.weighted_gap)[:n]
    raise ValueError(f"metric must be 'corr', 'gap', or 'weighted_gap', got {metric!r}")


def format_top_table(results: Sequence[Result], metric: str, n: int = 5) -> str:
    """Format a top-n results table as a string."""
    top = top_results(results, metric, n)
    labels = {
        "corr": "skill correlation",
        "gap": "mean skill gap",
        "weighted_gap": "weighted skill gap",
    }
    metric_label = labels[metric]
    width = 80
    lines = [
        f"Top {n} by {metric_label}",
        "=" * width,
        f"{'Rank':<6} {'Rating function':<28} {'skill_corr':>12} {'mean_gap':>12} {'weighted_gap':>12} {'rematches':>10}",
        "-" * width,
    ]
    for rank, r in enumerate(top, start=1):
        lines.append(
            f"{rank:<6} {r.label:<28} {r.corr:>+12.3f} {r.gap:>12.3f} {r.weighted_gap:>12.3f} {r.rematches:>10}"
        )
    return "\n".join(lines)


def present_top(results: Sequence[Result], cut_off: int = 5) -> tuple[list[Result], list[Result]]:
    """Print the top cut by correlation and by mean gap, then return both lists."""
    print(format_top_table(results, "corr", cut_off))
    print()
    print(format_top_table(results, "weighted_gap", cut_off))
    print()
    print(format_top_table(results, "gap", cut_off))
    return top_results(results, "corr", cut_off), top_results(results, "gap", cut_off)



In [ ]:
# move to tournament/study.py
def check_ctx(ctx: tuple[int, int, int]) -> bool:
    res = ctx[0] in {8, 32}
    res &= ctx[1] in {6, 7}
    res &= ctx[2] == 42
    return res

def shrink_ctx(ctx_list: list[tuple[int,int,int]]) -> list[tuple[int,int,int]]:
    return {ctx for ctx in ctx_list if check_ctx(ctx)}

small_ctxs = shrink_ctx(TOUR_CTX)
small_results = {ctx:result for ctx, result in big_results.items() if ctx in small_ctxs}

def present_result(ctx: tuple[int,int,int]) -> None:
    ctx_results = big_results[ctx]
    print(f"Players: {ctx[0]}, Rounds: {ctx[1]}, Seed: {ctx[2]}")
    print("-" * 40)
    present_top(ctx_results, 5)

def present_results(ctx_list: list[tuple[int,int,int]]) -> None:
    for ctx in ctx_list:
        present_result(ctx)
        print()
        
# present_results(small_ctxs)

present_result((32, 7, 42))

In [ ]:
# move to tournament/study.py
def count_top_appearances(
    results_by_context: dict[tuple[int, int, int], list[Result]],
    metric: str,
    cut_off: int = 5,
) -> Counter:
    """Count how many times each rating function appears in the top cut for a metric."""
    counts = Counter()
    for ctx_results in results_by_context.values():
        for result in top_results(ctx_results, metric, cut_off):
            counts[result.label] += 1
    return counts


def format_appearance_counts(counts: Counter, title: str) -> str:
    """Format a Counter of appearance counts as a readable table."""
    lines = [
        title,
        "=" * len(title),
        f"{'Rating function':<28} {'Appearances':>12}",
        "-" * 42,
    ]
    for label, count in counts.most_common():
        lines.append(f"{label:<28} {count:>12}")
    return "\n".join(lines[:4+8])

results = big_results # this line is a result of laziness
corr_counts = count_top_appearances(results, "corr", 5)
gap_counts = count_top_appearances(results, "gap", 5)
weighted_gap_counts = count_top_appearances(results, "weighted_gap", 5)

print(format_appearance_counts(corr_counts, "Top-5 by correlation"))
print()
print(format_appearance_counts(gap_counts, "Top-5 by mean gap"))
print()
print(format_appearance_counts(weighted_gap_counts, "Top-5 by weighted gap"))


In [ ]:
# move to tournament/study.py
# Legacy cell: top-5 per context with case tracking, now using the Result dataclass API.
corr_results = {}
gap_results = {}
weighted_gap_results = {}
top_cut = 8
for key, value in results.items():
    corr_results[key] = top_results(value, "corr", top_cut)
    gap_results[key] = top_results(value, "gap", top_cut)
    weighted_gap_results[key] = top_results(value, "weighted_gap", top_cut)

def _count_with_cases(top_by_context):
    counts = {}
    for key, results in top_by_context.items():
        for res in results:
            label = res.label
            if label in counts:
                counts[label]["count"] += 1
                counts[label]["cases"].append(key)
            else:
                counts[label] = {"count": 1, "cases": [key]}
    return counts

corr_counts = _count_with_cases(corr_results)
gap_counts = _count_with_cases(gap_results)
weighted_gap_counts = _count_with_cases(weighted_gap_results)
print(len(TOUR_CTX))
print(len(results.keys()))
corr_count_list = [(key, value["count"]) for key, value in corr_counts.items()]
corr_count_list.sort(key=lambda x: x[1], reverse=True)
for key, count in corr_count_list:
    print(f"{key} appears {count} times {count/len(TOUR_CTX)*100:.1f}%")

print()
gap_count_list = [(key, value["count"]) for key, value in gap_counts.items()]
gap_count_list.sort(key=lambda x: x[1], reverse=True)
for key, count in gap_count_list:
    print(f"{key} appears {count} times {count/len(results.keys())*100:.1f}%")

print()
weighted_gap_count_list = [(key, value["count"]) for key, value in weighted_gap_counts.items()]
weighted_gap_count_list.sort(key=lambda x: x[1], reverse=True)
for key, count in weighted_gap_count_list:
    print(f"{key} appears {count} times {count/len(results.keys())*100:.1f}% ")



## Final Standings per Rating Function

Each table below shows the final standings for one tournament run, sorted by win/loss record then rating score. The **Skill** column shows the player's true simulated skill so you can see how well the standings order reflects actual ability.

In [ ]:
# move to tournament/presentation.py
def show_standings(label, tour, rating_fn):
    records = compute_records(tour)
    key     = make_rank_key(rating_fn)
    ranked  = sorted(records.values(), key=key, reverse=True)
    # secondary sort: wins desc, then by key desc (already done above for tiebreaker)
    ranked  = sorted(ranked, key=lambda r: (-r.wins, -key(r)))

    sep = "-" * 66
    print(f"\n{'='*66}")
    print(f"  {label}")
    print(f"{'='*66}")
    print(f"  {'Rank':<5} {'Name':<8} {'Skill':>7} {'W':>4} {'L':>4} {'Diff':>6} {"Total:":>6} {'Score':>8}")
    print(f"  {sep}")
    for rank, r in enumerate(ranked, 1):
        score = key(r)
        print(f"  {rank:<5} {r.name:<8} {r.skill:>7} {r.wins:>4} {r.losses:>4} {r.agent_diff:>+6}  {r.agents_for:>6} {score:>8.3f}")

ctx = next(iter(results))
print(f"Players: {ctx[0]}, Rounds: {ctx[1]}, Seed: {ctx[2]}")
for res in results[first_ctx]:
    print(f"\n{res.label}")
    show_standings(res.label, res.tour, res.rating_fn)

#show_standings("Test", results[(10, 5, 42)][0].tour, results[(10, 5, 42)][0].rating_fn)

We can also ask the question of how often does one rating function capture the information of another. there are many ways to address this. One is to look at correlation of different pairs of rating functions.

## Rating Function Correlation

Do different rating functions rank players the same way? We compute pairwise
**Spearman rank correlations** between every pair of rating functions for the
same tournament context. A correlation near +1 means the two functions produce
nearly identical standings; near 0 means they disagree on player ordering.

We show two views:
1. **Single context** — one representative tournament (32 players, 7 rounds, seed 42).
2. **Averaged across all contexts** — the mean correlation across every
   `(n_players, n_rounds, seed)` context, giving a robust picture of which
   functions are redundant vs. complementary.

For each view we also compute the **score correlation** (Pearson on raw rating
scores, not just ranks) to detect functions that agree on order but disagree on
magnitude.


In [ ]:
# move to tournament/study.py
from tournament.correlation import (
    correlation_matrix,
    display_correlation_matrix,
    plot_correlation_heatmap,
)

# Pick a representative context with many players and rounds
demo_ctx = (32, 7, 42)
demo_results = big_results[demo_ctx]
labels = [res.label for res in demo_results]

# Rank correlation (Spearman-style)
rank_matrix = correlation_matrix(demo_results, mode="rank")
print(display_correlation_matrix(rank_matrix, labels, f"Rank correlation — {demo_ctx}"))
fig1 = plot_correlation_heatmap(rank_matrix, labels, f"Rank correlation — {demo_ctx}", mode="rank")
plt.show()

# Score correlation (Pearson on raw scores)
score_matrix = correlation_matrix(demo_results, mode="score")
print(display_correlation_matrix(score_matrix, labels, f"Score correlation — {demo_ctx}"))
fig2 = plot_correlation_heatmap(score_matrix, labels, f"Score correlation — {demo_ctx}", mode="score")
plt.show()


In [ ]:
# move to tournament/study.py
from collections import defaultdict

# Accumulate pairwise correlations across all contexts, then average
def average_correlation_matrices(results_by_ctx, mode="rank"):
    """Compute correlation_matrix for every context and average pairwise."""
    sums   = defaultdict(float)
    counts = defaultdict(int)
    for ctx, ctx_results in results_by_ctx.items():
        mat = correlation_matrix(ctx_results, mode=mode)
        for pair, val in mat.items():
            sums[pair] += val
            counts[pair] += 1
    return {pair: sums[pair] / counts[pair] for pair in sums}

avg_rank  = average_correlation_matrices(big_results, mode="rank")
avg_score = average_correlation_matrices(big_results, mode="score")

print(display_correlation_matrix(avg_rank, labels, "Averaged rank correlation — all contexts"))
fig3 = plot_correlation_heatmap(avg_rank, labels, "Averaged rank correlation — all contexts", mode="rank")
plt.show()

print(display_correlation_matrix(avg_score, labels, "Averaged score correlation — all contexts"))
fig4 = plot_correlation_heatmap(avg_score, labels, "Averaged score correlation — all contexts", mode="score")
plt.show()


### Interpretation

- **Rank correlation ≈ 1**: the two rating functions produce nearly identical
  standings orderings. They are redundant as tiebreakers.
- **Rank correlation ≈ 0**: the functions disagree on how to rank players within
  record groups. They capture different information.
- **Score correlation vs. rank correlation**: if score correlation is much lower
  than rank correlation, the functions agree on *order* but disagree on *how far
  apart* players are. This matters if the score magnitude feeds into pairing
  decisions.
- **Drop-off variants** (last k rounds) should correlate highly with their
  parent function when k ≥ n_rounds, and diverge as k decreases.
- **Averaged vs. single context**: if the averaged matrix looks similar to the
  single-context one, the correlations are stable across tournament sizes. If
  not, some functions may be more/less redundant depending on player count or
  round count.


# Legacy/Outdated

## Summary Comparison

We compare rating functions along two dimensions:

- **`skill_corr`** — Spearman-style rank correlation between final standings and true skill (+1 = perfect, 0 = random). Higher is better.
- **`mean_gap`** — average absolute skill difference between paired opponents (lower = better-matched pairings).
- **`rematches`** — number of repeat pairings across the tournament (lower = more variety).

The summary is shown twice: first sorted by `skill_corr` (descending), then sorted by `mean_gap` (ascending). For the top 5 in each ordering we show the tournament summary and a stacked round-by-round view of how each player's record evolved.


In [ ]:
# move to tournament/presentation.py
# Present the summary three ways: by correlation, mean gap, and weighted gap.
# For the top 5 in each ordering, show the tournament summary and a stacked
# round-by-round view.

# Recover the flat list of results and build a label -> rating_fn map.
result_list = [res for ctx_results in results.values() for res in ctx_results]
fn_by_label = dict(RATING_FNS)


def print_table(title, rows, key_name):
    print(f"\n{'='*80}")
    print(f"  {title}")
    print(f"{'='*80}")
    print(f"  {'Rank':<5} {'Rating function':<28} {key_name:>12} {'mean_gap':>10} {'weighted_gap':>10} {'rematches':>10}")
    print(f"  {'-'*73}")
    for rank, res in enumerate(rows, 1):
        print(
            f"  {rank:<5} {res.label:<28} {res.corr:>+12.3f} {res.gap:>10.3f} {res.weighted_gap:>10.3f} {res.rematches:>10}"
        )


def show_tournament(label, tour):
    rating_fn = fn_by_label.get(label)
    print(f"\n{'#'*80}")
    print(f"#  {label}")
    print(f"{'#'*80}")
    print(display_tournament_summary(tour))
    print()
    print(display_stacked_rounds(tour, show_groups=True, rating_fn=rating_fn))


# ── Top 5 by skill correlation ────────────────────────────────────────────────
top_corr = sorted(result_list, key=lambda r: r.corr, reverse=True)[:5]
print_table("Top 5 by skill correlation", top_corr, "skill_corr")

for res in top_corr:
    show_tournament(res.label, res.tour)

# ── Top 5 by mean skill gap ───────────────────────────────────────────────────
top_gap = sorted(result_list, key=lambda r: r.gap)[:5]
print_table("Top 5 by mean skill gap", top_gap, "skill_corr")

for res in top_gap:
    show_tournament(res.label, res.tour)

# ── Top 5 by weighted skill gap ───────────────────────────────────────────────
top_weighted_gap = sorted(result_list, key=lambda r: r.weighted_gap)[:5]
print_table("Top 5 by weighted skill gap", top_weighted_gap, "skill_corr")

for res in top_weighted_gap:
    show_tournament(res.label, res.tour)
